In [7]:
# -----------------------------
# Imports
# -----------------------------
import ssl
ssl._create_default_https_context = ssl._create_unverified_context

import torch
import pandas as pd
import matplotlib.pyplot as plt
from tqdm.auto import tqdm

import OpenAttack as oa
from transformers import BertTokenizer, BertForSequenceClassification
from datasets import load_dataset


device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# -----------------------------
# Tokenizer
# -----------------------------
tokenizer = BertTokenizer.from_pretrained("bert-base-uncased")

# -----------------------------
# Load model
# -----------------------------
model = BertForSequenceClassification.from_pretrained(
    "bert-base-uncased",
    num_labels=2
)

state_dict = torch.load("attentiondrop_sst2.pt", map_location=device)
model.load_state_dict(state_dict, strict=False)

model.to(device)
model.eval()

# -----------------------------
# Victim Wrapper
# -----------------------------
class MyClassifier(oa.Classifier):

    def get_pred(self, inputs):
        tokens = tokenizer(
            inputs,
            padding=True,
            truncation=True,
            max_length=128,
            return_tensors="pt"
        ).to(device)

        with torch.no_grad():
            outputs = model(**tokens)

        logits = outputs.logits
        preds = torch.argmax(logits, dim=1)

        return preds.cpu().numpy()

    def get_prob(self, inputs):
        tokens = tokenizer(
            inputs,
            padding=True,
            truncation=True,
            max_length=128,
            return_tensors="pt"
        ).to(device)

        with torch.no_grad():
            outputs = model(**tokens)

        probs = torch.softmax(outputs.logits, dim=1)

        return probs.cpu().numpy()


victim = MyClassifier()

# -----------------------------
# Load dataset
# -----------------------------
dataset = load_dataset("glue", "sst2")

samples = []

for i in range(50):
    text = dataset["validation"][i]["sentence"]
    label = int(dataset["validation"][i]["label"])

    samples.append({
        "x": text,
        "y": label
    })


# -----------------------------
# Attacker
# -----------------------------
attacker = oa.attackers.TextFoolerAttacker()

# -----------------------------
# Attack Evaluation
# -----------------------------
attack_eval = oa.AttackEval(attacker, victim)

print("\nStarting attacks...\n")

results = attack_eval.eval(samples)

# -----------------------------
# Convert results to DataFrame
# -----------------------------
df = pd.DataFrame(results)

print("\n===== Example Results =====\n")
print(df.head())

# -----------------------------
# Compute metrics
# -----------------------------
success_count = df["success"].sum()
total = len(df)

print("\n===== Attack Statistics =====")
print("Total Samples:", total)
print("Successful Attacks:", success_count)
print("Attack Success Rate:", success_count / total)

# -----------------------------
# Save CSV
# -----------------------------
df.to_csv("attack_results.csv", index=False)
print("\nResults saved to attack_results.csv")

# -----------------------------
# Visualization
# -----------------------------
labels = ["Successful", "Failed"]
values = [success_count, total - success_count]

plt.figure()
plt.bar(labels, values)
plt.title("Adversarial Attack Results")
plt.ylabel("Number of Samples")

plt.show()

Some weights of the model checkpoint at bert-base-uncased were not used when initializing BertForSequenceClassification: ['cls.predictions.transform.dense.bias', 'cls.seq_relationship.bias', 'cls.seq_relationship.weight', 'cls.predictions.transform.LayerNorm.weight', 'cls.predictions.transform.dense.weight', 'cls.predictions.transform.LayerNorm.bias', 'cls.predictions.bias']
- This IS expected if you are initializing BertForSequenceClassification from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing BertForSequenceClassification from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).
Some weights of BertForSequenceClassification were not initialized from the model checkpoint at bert-base-uncased and are newly i


Starting attacks...



ValueError: If using all scalar values, you must pass an index

In [10]:
import torch
import OpenAttack as oa
from transformers import BertTokenizer, BertForSequenceClassification
from datasets import load_dataset
# from OpenAttack.utils import DataInstance
# 1. Imports
import ssl
ssl._create_default_https_context = ssl._create_unverified_context
import torch
import pandas as pd
import OpenAttack as oa
from sklearn.model_selection import train_test_split
from torch.utils.data import DataLoader
from tqdm.auto import tqdm
from transformers import AutoTokenizer, AutoModelForSequenceClassification, AdamW, get_scheduler
from datasets import Dataset, DatasetDict
from OpenAttack.tags import TAG_English



device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# -----------------------------
# Load tokenizer
# -----------------------------
tokenizer = BertTokenizer.from_pretrained("bert-base-uncased")

# -----------------------------
# Load model
# -----------------------------
model = BertForSequenceClassification.from_pretrained(
    "bert-base-uncased",
    num_labels=2
)

state_dict = torch.load("attentiondrop_sst2.pt", map_location=device)
model.load_state_dict(state_dict, strict=False)

model.to(device)
model.eval()

# -----------------------------
# OpenAttack Victim Wrapper
# -----------------------------
class MyClassifier(oa.Classifier):

    def get_pred(self, inputs):
        tokens = tokenizer(
            inputs,
            padding=True,
            truncation=True,
            max_length=128,
            return_tensors="pt"
        ).to(device)

        with torch.no_grad():
            outputs = model(**tokens)

        logits = outputs.logits
        preds = torch.argmax(logits, dim=1)

        return preds.cpu().numpy()

    def get_prob(self, inputs):
        tokens = tokenizer(
            inputs,
            padding=True,
            truncation=True,
            max_length=128,
            return_tensors="pt"
        ).to(device)

        with torch.no_grad():
            outputs = model(**tokens)

        probs = torch.softmax(outputs.logits, dim=1)

        return probs.cpu().numpy()

victim = MyClassifier()

# -----------------------------
# Load SST2 dataset
# -----------------------------

dataset = load_dataset("glue", "sst2")

samples = []

for i in range(50):
    text = dataset["validation"][i]["sentence"]
    label = int(dataset["validation"][i]["label"])

    samples.append({
        "x": text,
        "y": label
    })

# -----------------------------
# Choose Attack Method
# -----------------------------
attacker = oa.attackers.TextFoolerAttacker()

# -----------------------------
# Attack Evaluation
# -----------------------------
attack_eval = oa.AttackEval(
    attacker,
    victim
)

print("\n==============================")
print("Starting Adversarial Attack")
print("==============================\n")

attack_eval.eval(samples)

Some weights of the model checkpoint at bert-base-uncased were not used when initializing BertForSequenceClassification: ['cls.predictions.transform.dense.bias', 'cls.seq_relationship.bias', 'cls.seq_relationship.weight', 'cls.predictions.transform.LayerNorm.weight', 'cls.predictions.transform.dense.weight', 'cls.predictions.transform.LayerNorm.bias', 'cls.predictions.bias']
- This IS expected if you are initializing BertForSequenceClassification from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing BertForSequenceClassification from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).
Some weights of BertForSequenceClassification were not initialized from the model checkpoint at bert-base-uncased and are newly i


Starting Adversarial Attack



{'Total Attacked Instances': 50,
 'Successful Instances': 32,
 'Attack Success Rate': 0.64,
 'Avg. Running Time': 0.15256745338439942,
 'Total Query Exceeded': 0.0,
 'Avg. Victim Model Queries': 57.8}

In [11]:
attackers = [
    oa.attackers.DeepWordBugAttacker(),
    oa.attackers.UATAttacker()
]

attacker_names = [
    "DeepWordBug",
    "UATA"
]

In [12]:
# Run evaluation
results_all = {}
for name, attacker in zip(attacker_names, attackers):
    print(f"\n=== Evaluating attacker: {name} ===")
    evaluator = oa.AttackEval(attacker, victim)
    result = evaluator.eval(samples, visualize=True)
    results_all[name] = result
    print(f"{name} results:", result)



=== Evaluating attacker: DeepWordBug ===
Sample: 1 =====================================================================
Label: 1 (99.97%) --> 0 (69.57%)            |                                   
                                            |                                   
it ' s a charming  and often affecting      | Running Time:            0.0080478
it ` s a charｍing and often affectіng      | Query Exceeded:          no       
                                            | Victim Model Queries:    13       
journey .                                   | Succeed:                 yes      
joսrney .                                   |                                   
                                            |                                   
Sample: 2 =====================================================================
Label: 0 (98.04%) --> Failed!               |                                   
                                            | Running Time:           